In [11]:
import pandas as pd
import numpy as np

data={
    'Weather':
    ['sunny','sunny','cloudy','rainy','rainy','rainy','cloudy','sunny','sunny','rainy',
    'sunny','rainy','sunny','cloudy'],
    'Temperature':['hot','hot','hot','mild','cool','cool','cool','mild','cool',
                     'mild','mild','mild','hot','mild'],
    'SoilMoisture':['dry','dry','dry','dry','moist','moist','moist','dry','moist','moist',
                    'moist','dry','moist','dry'],
    'Wind':['weak','strong','weak','weak','weak','strong','strong',
            'weak','weak','weak','strong','strong','weak','strong'],
    'Irrigate':['no','no','yes','yes','yes','no','yes','no','yes','yes','yes','yes','yes','no']
    }

df = pd.DataFrame(data)

# Step 1: Define Entropy function
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)

    prob = counts / np.sum(counts)
    entropy_val = -np.sum(prob * np.log2(prob + 1e-9))  # Avoid log(0)
    return entropy_val

# Step 2: Information Gain
def information_gain(data, feature, target_name):
    total_entropy = entropy(data[target_name])
    values, counts = np.unique(data[feature], return_counts=True)
    weighted_entropy = 0
    for i in range(len(values)):
        subset = data[data[feature] == values[i]]

        if not subset.empty:
            weighted_entropy += (counts[i]/np.sum(counts)) * entropy(subset[target_name])
    return total_entropy - weighted_entropy

# Step 3: ID3 Algorithm
def id3(data, features, target_name, depth=0):
  indent = "  " * depth


  if len(np.unique(data[target_name])) == 1:
       return np.unique(data[target_name])[0]

  if len(features) == 0:
       return data[target_name].mode()[0]

  print(f"\n{indent}Evaluating features at depth {depth}:")
  gains = []
  for feature_item in features:
    gain = information_gain(data, feature_item, target_name)
    gains.append(gain)
    print(f"{indent}  IG({feature_item}) = {gain:.4f}")


  if not gains or np.all(np.array(gains) == 0):
      return data[target_name].mode()[0]

  best_feature = features[np.argmax(gains)]
  print(f"{indent}Best Feature = {best_feature}")
  tree = {best_feature: {}}

  for value in np.unique(data[best_feature]):
    print(f"{indent}Splitting {best_feature} = {value}")
    subset = data[data[best_feature] == value]
    if subset.shape[0] == 0:

      tree[best_feature][value] = data[target_name].mode()[0]
    else:
      remaining_features = [f for f in features if f != best_feature]
      subtree = id3(subset, remaining_features, target_name, depth + 1)
      tree[best_feature][value] = subtree
  return tree

def print_tree(tree, indent=""):
  if not isinstance(tree, dict):
    print(indent + "- " + str(tree))
    return
  for feature, branches in tree.items():
    for value, subtree in branches.items():
      print(f"{indent}{feature} = {value}:")
      print_tree(subtree, indent + "  ")

features = list(df.columns[:-1])
tree_model = id3(df, features, "Irrigate")
print("\nFinal Decision Tree:\n")
print_tree(tree_model)



Evaluating features at depth 0:
  IG(Weather) = 0.0571
  IG(Temperature) = 0.0292
  IG(SoilMoisture) = 0.1518
  IG(Wind) = 0.0481
Best Feature = SoilMoisture
Splitting SoilMoisture = dry

  Evaluating features at depth 1:
    IG(Weather) = 0.6995
    IG(Temperature) = 0.0202
    IG(Wind) = 0.0202
  Best Feature = Weather
  Splitting Weather = cloudy

    Evaluating features at depth 2:
      IG(Temperature) = 1.0000
      IG(Wind) = 1.0000
    Best Feature = Temperature
    Splitting Temperature = hot
    Splitting Temperature = mild
  Splitting Weather = rainy
  Splitting Weather = sunny
Splitting SoilMoisture = moist

  Evaluating features at depth 1:
    IG(Weather) = 0.1981
    IG(Temperature) = 0.1281
    IG(Wind) = 0.1981
  Best Feature = Weather
  Splitting Weather = cloudy
  Splitting Weather = rainy

    Evaluating features at depth 2:
      IG(Temperature) = 0.2516
      IG(Wind) = 0.9183
    Best Feature = Wind
    Splitting Wind = strong
    Splitting Wind = weak
  Splitti